# Метод 3 — LLM-судья: от холистического промпта к разделённым осям, self-consistency и fine-tuning

**Задача.** Вход — `(вопрос из диалога, до 8 retrieved-чанков, ответ бота)`.
Выход — `faithfulness ∈ {0,1}` (ответ подкреплён контекстом) и
`relevance ∈ {0,1}` (ответ отвечает на вопрос); целевая метка

```
reliable = faithfulness AND relevance
```

Первичная метрика — **macro-F1 по `reliable`** с 95% доверительным интервалом,
протокол 5×5 CV по `data/splits/folds_alfa.json`. Корпус — 2233 кейса, 72.4% reliable.

**Метод 3** — LLM-судья: модель читает кейс и выносит вердикт. Вероятность вердикта
берётся из logprobs токена `PASS`/`FAIL`, поэтому метод даёт непрерывный скор, а не
только бинарный ответ, и порог подбирается протоколом внутри train-части фолда.

---

## Что уже известно (базовая линия, holistic-промпт)

| Показатель | Значение | Чем плохо |
|---|---:|---|
| AUC(`p_faith`) против faithfulness | 0.627 | слабо, но сигнал есть |
| **AUC(`p_rel`) против relevance** | **0.497** | сигнала нет вообще |
| Доля вердиктов `RELEVANCE: PASS` | 98.6–100% | при золотой доле 84.9% — судья почти всегда говорит «да» |
| Доля `p_faith > 0.99` | 50–79% | насыщение: ранжировать нечего |
| Recall класса unreliable | 0.22–0.33 | пропускает две трети плохих ответов |

Из этого диагноза выведены гипотезы ноутбука. Каждая проверяется отдельным прогоном
по всему корпусу и своей ячейкой метрик.

## Гипотезы

| # | Гипотеза | Что меняем | Целевое число | Время на A100 |
|---|---|---|---|---:|
| **Г0** | база | один вызов, оба вердикта | — | ~15 мин |
| **Г1** | оси мешают друг другу | два независимых вызова; промпт relevance **не получает чанков** | AUC(`p_rel`) ≥ 0.60, доля `RELEVANCE: PASS` ≤ 92% | ~20 мин |
| **Г2** | одиночный сэмпл насыщен | self-consistency k=8, усредняем **вероятности**, а не голоса | доля `p_faith>0.99` < 30% | ~50 мин |
| **Г3** | длинный контекст размывает вердикт | судим каждый чанк отдельно, фича `chunk_disagreement` | recall unreliable ≥ 0.45 при precision ≥ 0.55 | ~40 мин |
| **H5** | маркеры как текстовый feedback помогают промпт-оптимизации | GEPA: `markers` против `plain`, 3 сида, равный бюджет | верх 95% ДИ приращения > 0 | ~7 ч (Jobs) |
| **H1** | fine-tuned судья превосходит prompt-optimized | LoRA на Qwen2.5-7B, тот же backbone и та же схема logprobs | Δ macro-F1 > 0 с ДИ, не пересекающим ноль | ~7.5 ч (Jobs) |

H5 и H1 длиннее двух часов и запускаются через DataSphere Jobs: VM ноутбука
останавливается при простое. Ячейки Г0–Г3 исполняются здесь.

## Как читать ноутбук

Ноутбук **самостоятельный**: логика метода — промпты, извлечение вероятностей,
диагностика вырождения, метрики — видна прямо в ячейках. Корпусные прогоны идут
через CLI репозитория (`scripts/run_m3.py`, `scripts/evaluate_cv.py`): они
резюмируемы, кэшируют вызовы судьи и кладут `run.yaml` с git-хэшем рядом с каждым
артефактом. Разбиение читается **только** из `folds_alfa.json`.

Порядок: **Run All** до раздела H5, затем отправить два задания в Jobs и вернуться к
итоговой таблице, когда они досчитают.

## Config

In [ ]:
# ======================= КОНФИГУРАЦИЯ — правится только здесь =======================
BASE      = "/home/jupyter/filestore/neurodrive"  # File Storage: переживает рестарт VM
REPO      = f"{BASE}/rag-reliability"
REPO_URL  = "https://github.com/MurkaSelebry/rag-reliability.git"
BRANCH    = "main"                    # актуальная ветка: волны 1-4 влиты, main == integration
CACHE     = f"{BASE}/cache/m3_judge"  # кэш судьи -> любой прогон резюмируется
LOGS      = f"{BASE}/logs"

DATA      = "data/alfa.jsonl"               # канонический корпус, 2233 кейса
FOLDS     = "data/splits/folds_alfa.json"   # единственный источник разбиения
OUT       = "predictions/alfa/m3_judge"     # <вариант>/{scores.jsonl,report.json,run.yaml}

MODEL     = "Qwen/Qwen2.5-7B-Instruct"
API_BASE  = "http://localhost:8000/v1"
CONCURRENCY = 16
SC_N, SC_TEMPERATURE = 8, 0.8          # self-consistency: k сэмплов на ось
SEED      = 42

# Целевые числа гипотез — из docs/specs/30_PHASE2_метод3.md; печатаются в вердиктах.
TARGET_AUC_REL          = 0.60   # Г1: AUC(p_rel) сейчас 0.497
TARGET_RELEVANCE_PASS   = 0.92   # Г1: сейчас 0.986-1.00 при золотой доле 0.849
TARGET_SATURATED_SHARE  = 0.30   # Г2: доля p_faith > 0.99, сейчас 0.50-0.79
TARGET_RECALL_UNRELIABLE = 0.45  # Г3: сейчас 0.22-0.33
TARGET_PRECISION_UNRELIABLE = 0.55
# ===================================================================================

import os, subprocess, sys, json, time

os.environ["HF_HOME"] = f"{BASE}/hf"          # до импорта transformers: веса ~15 GB
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ.setdefault("OPENAI_API_KEY", "dummy")  # vLLM ключ не проверяет, клиент требует
for directory in (LOGS, CACHE, f"{BASE}/hf"):
    os.makedirs(directory, exist_ok=True)
print("конфигурация принята")

## Setup — стек, репозиторий, железо

Драйвер DataSphere — CUDA 12.2. `torch` с PyPI (cu13) падает «driver too old»,
базовый `torch 2.0.1/cu118` слишком стар для актуального `trl`. Отсюда пин
`torch==2.5.1` на cu121; `vllm` подобран под ровно эту версию torch — менять одно
без другого нельзя. `numpy==1.26.4` ставится **последним**: numpy 2 ломает
C-расширения базового образа.

После первого прогона ячейки установки — **Kernel → Restart** и снова Run All.

In [ ]:
if not os.path.isdir(REPO):
    subprocess.check_call(["git", "clone", "-b", BRANCH, REPO_URL, REPO])
else:
    subprocess.check_call(["git", "-C", REPO, "fetch", "origin", BRANCH])
    subprocess.check_call(["git", "-C", REPO, "checkout", BRANCH])
    subprocess.check_call(["git", "-C", REPO, "pull", "--ff-only"])

# Идентичность нужна, чтобы коммитить артефакты прямо из ноутбука.
subprocess.check_call(["git", "-C", REPO, "config", "user.name", "datasphere-runner"])
subprocess.check_call(["git", "-C", REPO, "config", "user.email", "datasphere@localhost"])

os.chdir(REPO)
sys.path.insert(0, os.path.join(REPO, "src"))
head = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"]).decode().strip()
print(f"репозиторий {REPO} на {BRANCH} @ {head}")

In [ ]:
pip = lambda *args: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

pip("torch==2.5.1", "torchvision==0.20.1", "torchaudio==2.5.1",
    "--index-url", "https://download.pytorch.org/whl/cu121")
pip("vllm==0.6.6.post1")                  # собран под torch 2.5.1
pip("-e", f"{REPO}[cloud,gepa]")          # клиент судьи + DSPy для GEPA
pip("scikit-learn", "pandas")             # метрики и таблицы в ячейках этого ноутбука
pip("numpy==1.26.4")                      # ПОСЛЕДНИМ: перекрывает numpy 2, если он подтянулся
print("стек установлен; если torch уже импортировался в этом ядре — Kernel → Restart и Run All")

In [ ]:
import torch, psutil

n_gpu = torch.cuda.device_count()
vram = torch.cuda.get_device_properties(0).total_memory / 1e9 if n_gpu else 0.0
print(f"GPU {torch.cuda.get_device_name(0) if n_gpu else '—'} | VRAM {vram:.0f}GB | "
      f"RAM {psutil.virtual_memory().total / 1e9:.0f}GB | GPUs {n_gpu}")

assert n_gpu >= 1, (
    "выбрана CPU-конфигурация. Судья 7B требует GPU: Ресурсы проекта → "
    "Конфигурация вычислительных ресурсов → g2.1 → рестарт VM"
)
assert vram >= 70, (
    f"VRAM {vram:.0f} GB < 70 GB. Контур рассчитан на g2.1 (1× A100 80 GB): на V100 32 GB "
    "нет bf16, а Qwen2.5-7B с контекстом 8192 не помещается вместе с KV-кэшем"
)

## Data

Корпус и разбиение читаются как есть. `split_samples` в этом ноутбуке не вызывается
нигде: стратифицированный сплит давал утечку 24.9% по вопросу (один и тот же вопрос
попадал и в train, и в test), и полученные им числа несравнимы с числами на
group-aware фолдах.

In [ ]:
from rag_reliability.dataset import load_jsonl
from rag_reliability.methods.surface.features import split_chunks

samples = load_jsonl(DATA)
folds = json.load(open(FOLDS, encoding="utf-8"))

n = len(samples)
n_reliable = sum(s.reliable for s in samples)
print(f"корпус: {n} кейсов, reliable {n_reliable} ({n_reliable / n:.1%})")
print(f"faithfulness=1: {sum(s.faithfulness for s in samples) / n:.1%}   "
      f"relevance=1: {sum(s.relevance for s in samples) / n:.1%}")
print(f"чанков на кейс: медиана {sorted(len(split_chunks(s.context)) for s in samples)[n // 2]}")

cfg, stats = folds["config"], folds["stats"]
print(f"\nфолды: {cfg['n_folds']}×{cfg['n_repeats']} CV, seed {cfg['seed']}, "
      f"групп {stats['n_groups']}, исключено гигантской группой {stats['excluded_ids']} id")
print(f"утечка после group-aware сплита: {stats['leak_check']}")

In [ ]:
# Один кейс целиком: судья видит ровно это.
sample = samples[20]
chunks = split_chunks(sample.context)
print("ВОПРОС (хвост диалога):\n", sample.question[-600:], "\n")
print(f"КОНТЕКСТ: {len(chunks)} чанк(ов), первый:\n", chunks[0][:600], "\n")
print("ОТВЕТ БОТА:\n", sample.answer, "\n")
print(f"ЗОЛОТО: faithfulness={sample.faithfulness}  relevance={sample.relevance}  "
      f"reliable={int(sample.reliable)}  marker={sample.marker}")

## Метрики: как считается каждое число в ноутбуке

Две вспомогательные функции на весь ноутбук. `report_row` вытаскивает строку сводной
таблицы из `report.json`, который пишет `scripts/evaluate_cv.py` (5×5 CV, порог
подбирается внутри train-части фолда, бутстрэп B=10000, 500 нулевых прогонов).
`judge_diagnostics` считает по `scores.jsonl` то, из-за чего базовый прогон и был
признан непригодным: насыщение вероятностей и вырождение оси relevance в «всегда PASS».

Число без интервала в этом проекте не считается результатом, поэтому AUC тоже идёт с
бутстрэп-ДИ.

In [ ]:
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, classification_report
from rag_reliability.evaluation.bootstrap import bootstrap_ci


def auc_ci(labels, values, B=2000, seed=SEED):
    """ROC-AUC с перцентильным бутстрэп-ДИ по кейсам."""
    labels, values = np.asarray(labels, dtype=int), np.asarray(values, dtype=float)
    metric = lambda y, x: 0.5 if len(set(y.tolist())) < 2 else float(roc_auc_score(y, x))
    r = bootstrap_ci(labels, values, metric, B=B, seed=seed)
    return r.point, r.lo, r.hi


def load_scores(path):
    """{id: scores} из scores.jsonl прогона."""
    rows = {}
    for line in open(path, encoding="utf-8"):
        if line.strip():
            row = json.loads(line)
            rows[str(row["id"])] = row["scores"]
    assert rows, f"{path}: пустой артефакт"
    return rows


def report_row(path, label):
    """Строка сводной таблицы из report.json."""
    rep = json.load(open(path, encoding="utf-8"))
    primary, conf = rep["primary"], rep["operational"]["confusion"]
    # unreliable — положительный класс операционной задачи: его и ищет quality gate.
    recall = conf["tp"] / (conf["tp"] + conf["fn"]) if conf["tp"] + conf["fn"] else 0.0
    precision = conf["tp"] / (conf["tp"] + conf["fp"]) if conf["tp"] + conf["fp"] else 0.0
    # Поосевые F1 есть только там, где прогон дал обе оси и в evaluate_cv переданы
    # --faith-expr/--rel-expr. NaN здесь означает «не считалось», а не «ноль»:
    # молчаливый 0.0 в сводной таблице читался бы как провал метода.
    axes_f1 = rep["axes"]
    return {
        "вариант": label,
        "macro-F1": round(primary["value"], 4),
        "95% ДИ": f'[{primary["ci95"][0]:.4f}; {primary["ci95"][1]:.4f}]',
        "выше шума": primary["above_noise"],
        "F1 faith": (round(axes_f1["faithfulness_f1_macro"]["value"], 4)
                     if "faithfulness_f1_macro" in axes_f1 else float("nan")),
        "F1 rel": (round(axes_f1["relevance_f1_macro"]["value"], 4)
                   if "relevance_f1_macro" in axes_f1 else float("nan")),
        "ROC-AUC": round(rep["operational"]["roc_auc"], 4),
        "recall unrel": round(recall, 3),
        "precision unrel": round(precision, 3),
        "n": rep["protocol"]["n_evaluated"],
    }


def judge_diagnostics(scores_path, label):
    """Вырождение судьи: насыщение p_faith и «всегда PASS» по оси relevance.

    Пофрагментный прогон вероятностей осей не даёт вовсе (ось relevance чанков не
    получает), поэтому наличие ключей проверяется явно, а не подставляется дефолт.
    """
    rows = load_scores(scores_path)
    evaluated = [s for s in samples if s.id in rows]
    assert evaluated, f"{scores_path}: ни одного пересечения с корпусом"
    keys = next(iter(rows.values()))
    out = {"вариант": label, "n": len(evaluated)}

    if "m3.p_faith" in keys:
        p_faith = [rows[s.id]["m3.p_faith"] for s in evaluated]
        point, lo, hi = auc_ci([s.faithfulness for s in evaluated], p_faith)
        out["AUC p_faith"] = f"{point:.3f} [{lo:.3f}; {hi:.3f}]"
        out["доля p_faith>0.99"] = round(float(np.mean(np.asarray(p_faith) > 0.99)), 3)
    elif "m3.max_chunk_score" in keys:   # пофрагментный путь: лучший чанк вместо p_faith
        best_chunk = [rows[s.id]["m3.max_chunk_score"] for s in evaluated]
        point, lo, hi = auc_ci([s.faithfulness for s in evaluated], best_chunk)
        out["AUC p_faith"] = f"{point:.3f} [{lo:.3f}; {hi:.3f}] (max_chunk_score)"
        out["доля p_faith>0.99"] = round(float(np.mean(np.asarray(best_chunk) > 0.99)), 3)

    if "m3.p_rel" in keys:
        p_rel = [rows[s.id]["m3.p_rel"] for s in evaluated]
        point, lo, hi = auc_ci([s.relevance for s in evaluated], p_rel)
        out["AUC p_rel"] = f"{point:.3f} [{lo:.3f}; {hi:.3f}]"
        # Вердикт PASS = вероятность выше 0.5: именно так его читает парсер судьи.
        out["доля RELEVANCE PASS"] = round(float(np.mean(np.asarray(p_rel) > 0.5)), 3)
    return out


# label -> {"scores", "report", "score"}; score считает итоговый скор кейса из его
# фич — тем же выражением, которое передано в evaluate_cv через --score-expr.
JOINT_SCORE = lambda sc: sc["m3.p_faith"] * sc["m3.p_rel"]
RUNS = {}
print("метрики готовы; золотая доля relevance=1 в корпусе:",
      f"{sum(s.relevance for s in samples) / len(samples):.3f}")

## vLLM и обязательный смоук на logprobs

`--max-logprobs 25` обязателен: клиент судьи запрашивает `top_logprobs=20`, и без
этого флага сервер вернёт вердикт без вероятностей — метод деградирует в
regex-парсинг бинарного ответа.

Смоук пропускать нельзя. Извлечение вердикта чувствительно к тому, как токенизатор
режет `PASS`/`FAIL`: при односимвольном первом подтокене вероятность **молча**
становится 0.5 у всех кейсов. Прогон на 2233 кейса при этом выглядит успешным, а
сигнала в нём нет — ровно этот отказ и стоил ветке одного полного прогона.

In [ ]:
import requests

subprocess.Popen(
    f"vllm serve {MODEL} --port 8000 --max-model-len 8192 "
    f"--gpu-memory-utilization 0.85 --max-logprobs 25 > {LOGS}/vllm.log 2>&1",
    shell=True,
)
for _ in range(120):                     # до 10 минут: первая загрузка весов долгая
    try:
        if requests.get(f"{API_BASE}/models", timeout=2).ok:
            print("vLLM up")
            break
    except Exception:
        pass
    time.sleep(5)
else:
    raise RuntimeError(
        f"vLLM не поднялся за 10 минут — смотри {LOGS}/vllm.log. Типовая причина: не "
        "хватило VRAM под KV-кэш; понизить --gpu-memory-utilization до 0.75 и "
        "--max-model-len до 4096"
    )

In [ ]:
# 5 кейсов, ~10 секунд.
!python scripts/score.py --method m3_openai_judge --variant zero_shot \
    --data {DATA} --limit 5 --model {MODEL} \
    --m3-api-base {API_BASE} --m3-cache-dir {BASE}/cache/smoke \
    --output {BASE}/smoke/scores.jsonl

In [ ]:
smoke = [json.loads(line) for line in open(f"{BASE}/smoke/scores.jsonl", encoding="utf-8")]
print("prob_method:", [row["prob_method"] for row in smoke])
print("m3.p_faith :", [round(row["scores"]["m3.p_faith"], 4) for row in smoke])

assert smoke, "смоук не дал ни одной строки — смотри вывод предыдущей ячейки"
assert all(row["prob_method"] == "logprobs" for row in smoke), (
    "PASS/FAIL режется на подтокены: извлечение вероятностей вырождается в 0.5 для всех "
    "кейсов. Полный прогон НЕ запускать. prob_method == 'regex' на всех строках — другой "
    "симптом: сервер не отдаёт logprobs, проверить --max-logprobs 25 у vLLM"
)
assert len({round(row["scores"]["m3.p_faith"], 3) for row in smoke}) > 1, (
    "все вероятности одинаковы — извлечение сломано"
)
print("\nlogprobs smoke OK — можно запускать корпусные прогоны")

## Что именно видит судья

Промпты — не строки в коде, а версионированные файлы `configs/prompts/*.yaml`: их
текст дословно повторяет шкалы из `from_organizators/readme.md`. Ниже — ровно то, что
уходит в модель для одного кейса по каждой оси.

Ключевая деталь оси relevance: в её промпте **нет чанков**. Релевантность — это
отношение «вопрос ↔ ответ», и контекст в этом вопросе только шумит. Именно
совместный промпт с чанками и давал AUC 0.497.

In [ ]:
from rag_reliability.methods.m3 import axes

for axis in axes.AXES:
    spec = axes.load_axis_prompt(axis)
    system, user = axes.build_axis_prompt(sample, axis, mode="zero_shot")
    print("=" * 80)
    print(f"ОСЬ {axis.upper()}   (версия промпта {spec.version}, чанки в промпте: {spec.needs_context})")
    print("=" * 80)
    print("--- SYSTEM ---\n", system[:1500])
    print("\n--- USER (обрезано) ---\n", user[:800])
    print()

In [ ]:
# Чеклист маркеров, который подмешивается в промпт faithfulness: 13 кодов ошибок
# организаторов. Судья обязан назвать код прежде, чем вынести вердикт, — вердикт
# без названной причины оказывался «PASS по умолчанию».
glosses = axes.load_markers()
print(axes.build_marker_checklist(sorted(glosses), glosses))

## Г0 · База — холистический промпт

Один вызов на кейс, оба вердикта в одном ответе. Это состояние, с которого ветка
начиналась; все последующие прогоны сравниваются с ним.

In [ ]:
# ~15 мин на 2233 кейсах. Прогон резюмируем: при обрыве повторить ту же команду.
!python scripts/run_m3.py --data {DATA} \
    --output {OUT}/joint/scores.jsonl --run-meta {OUT}/joint/run.yaml \
    --mode zero_shot --backend openai_judge --prompt-style joint \
    --model {MODEL} --api-base {API_BASE} \
    --cache-dir {CACHE}/joint --concurrency {CONCURRENCY}

In [ ]:
!python scripts/evaluate_cv.py --data {DATA} --folds {FOLDS} \
    --scores {OUT}/joint/scores.jsonl \
    --score-expr "m3.p_faith * m3.p_rel" \
    --faith-expr "m3.p_faith" --rel-expr "m3.p_rel" \
    --output {OUT}/joint/report.json

In [ ]:
RUNS["Г0 holistic"] = {"scores": f"{OUT}/joint/scores.jsonl",
                       "report": f"{OUT}/joint/report.json", "score": JOINT_SCORE}
display(pd.DataFrame([report_row(RUNS["Г0 holistic"]["report"], "Г0 holistic")]))
display(pd.DataFrame([judge_diagnostics(RUNS["Г0 holistic"]["scores"], "Г0 holistic")]))

## Г1 · Разделение осей

**Гипотеза.** Оси мешают друг другу в одном вызове: длинный контекст, нужный для
faithfulness, вытесняет из внимания вопрос, нужный для relevance. Два независимых
вызова с промптом relevance **без чанков** должны вернуть оси сигнал.

**Целевые числа.** AUC(`p_rel`) ≥ 0.60 (было 0.497); доля вердиктов
`RELEVANCE: PASS` ≤ 0.92 (было 0.986–1.00 при золотой доле 0.849).

In [ ]:
# ~20 мин: вызова два, но промпт relevance втрое короче — чанков в нём нет.
!python scripts/run_m3.py --data {DATA} \
    --output {OUT}/axes/scores.jsonl --run-meta {OUT}/axes/run.yaml \
    --mode zero_shot --backend openai_judge --prompt-style axes \
    --model {MODEL} --api-base {API_BASE} \
    --cache-dir {CACHE}/axes --concurrency {CONCURRENCY}

In [ ]:
!python scripts/evaluate_cv.py --data {DATA} --folds {FOLDS} \
    --scores {OUT}/axes/scores.jsonl \
    --score-expr "m3.p_faith * m3.p_rel" \
    --faith-expr "m3.p_faith" --rel-expr "m3.p_rel" \
    --compare {OUT}/joint/scores.jsonl \
    --output {OUT}/axes/report.json

In [ ]:
RUNS["Г1 axes"] = {"scores": f"{OUT}/axes/scores.jsonl",
                   "report": f"{OUT}/axes/report.json", "score": JOINT_SCORE}
diag = judge_diagnostics(RUNS["Г1 axes"]["scores"], "Г1 axes")
display(pd.DataFrame([report_row(RUNS["Г1 axes"]["report"], "Г1 axes")]))
display(pd.DataFrame([diag]))

auc_rel = float(diag["AUC p_rel"].split()[0])
pass_share = diag["доля RELEVANCE PASS"]
print(f"\nГ1: AUC(p_rel) {auc_rel:.3f} против цели {TARGET_AUC_REL} — "
      f"{'ЦЕЛЬ ВЗЯТА' if auc_rel >= TARGET_AUC_REL else 'цель не взята'}")
print(f"Г1: доля RELEVANCE PASS {pass_share:.3f} против цели ≤{TARGET_RELEVANCE_PASS} — "
      f"{'ЦЕЛЬ ВЗЯТА' if pass_share <= TARGET_RELEVANCE_PASS else 'цель не взята'}")
print("Нижняя граница ДИ у AUC важнее точечной оценки: она и решает, отличим ли сигнал от нуля.")

## Г2 · Self-consistency k=8

**Гипотеза.** Одиночный сэмпл при температуре 0 даёт насыщенную вероятность: половина
кейсов получает `p_faith > 0.99`, и ранжировать их нечем. Восемь сэмплов при T=0.8
размывают насыщение и дают вдобавок меру согласия — `p_std`.

Усредняются **вероятности**, а не голоса: голосование выбрасывает ровно ту
информацию, ради которой брались logprobs. `p_vote` считается рядом как диагностика.

**Целевое число.** Доля `p_faith > 0.99` < 0.30 (было 0.50–0.79).

Prompt-часть переиспользуется prefix-кэшем vLLM, дорог только выход — отсюда ×3.5 ко
времени, а не ×8.

In [ ]:
# ~50 мин. Резюмируется через {CACHE}/axes_sc8 — при обрыве повторить ту же команду.
!python scripts/run_m3.py --data {DATA} \
    --output {OUT}/axes_sc8/scores.jsonl --run-meta {OUT}/axes_sc8/run.yaml \
    --mode zero_shot --backend openai_judge --prompt-style axes \
    --sc-n {SC_N} --sc-temperature {SC_TEMPERATURE} \
    --model {MODEL} --api-base {API_BASE} \
    --cache-dir {CACHE}/axes_sc8 --concurrency {CONCURRENCY}

In [ ]:
!python scripts/evaluate_cv.py --data {DATA} --folds {FOLDS} \
    --scores {OUT}/axes_sc8/scores.jsonl \
    --score-expr "m3.p_faith * m3.p_rel" \
    --faith-expr "m3.p_faith" --rel-expr "m3.p_rel" \
    --compare {OUT}/axes/scores.jsonl \
    --output {OUT}/axes_sc8/report.json

In [ ]:
RUNS["Г2 axes+SC8"] = {"scores": f"{OUT}/axes_sc8/scores.jsonl",
                       "report": f"{OUT}/axes_sc8/report.json", "score": JOINT_SCORE}
diag = judge_diagnostics(RUNS["Г2 axes+SC8"]["scores"], "Г2 axes+SC8")
display(pd.DataFrame([report_row(RUNS["Г2 axes+SC8"]["report"], "Г2 axes+SC8")]))
display(pd.DataFrame([diag]))

saturated = diag["доля p_faith>0.99"]
print(f"\nГ2: доля p_faith>0.99 {saturated:.3f} против цели <{TARGET_SATURATED_SHARE} — "
      f"{'ЦЕЛЬ ВЗЯТА' if saturated < TARGET_SATURATED_SHARE else 'цель не взята'}")

# Разброс между сэмплами — самостоятельная фича: судья, который расходится сам с собой,
# ошибается чаще. Проверяем, есть ли в нём сигнал помимо самой вероятности.
sc_rows = load_scores(RUNS["Г2 axes+SC8"]["scores"])
evaluated = [s for s in samples if s.id in sc_rows]
for std_key in ("m3.p_faith_std", "m3.p_rel_std"):
    point, lo, hi = auc_ci([1 - int(s.reliable) for s in evaluated],
                           [sc_rows[s.id][std_key] for s in evaluated])
    print(f"AUC({std_key}) против unreliable: {point:.3f} [{lo:.3f}; {hi:.3f}]")

## Г3 · Пофрагментная верификация

**Гипотеза.** В холистическом промпте судья видит до 8 чанков разом; ошибка типа
`reason_chunk_fact_mixup` (факт склеен из двух разных чанков) в такой постановке
невидима. Если судить каждый чанк отдельно, появляется `chunk_disagreement` —
расхождение вердиктов между чанками, прямой детектор склейки.

Ось только faithfulness: промпт relevance чанков не получает по построению.
Итоговые фичи: `m3.max_chunk_score`, `m3.mean_chunk_score`, `m3.chunk_disagreement`,
`m3.n_supporting`, `m3.argmax_chunk`.

**Целевое число.** recall класса unreliable ≥ 0.45 при precision ≥ 0.55
(было 0.22–0.33).

In [ ]:
# ~40 мин: 8 коротких промптов на кейс вместо одного длинного, prefix-кэш снимает общую часть.
!python scripts/run_m3.py --data {DATA} \
    --output {OUT}/perchunk/scores.jsonl --run-meta {OUT}/perchunk/run.yaml \
    --mode zero_shot --backend openai_judge --prompt-style perchunk \
    --model {MODEL} --api-base {API_BASE} \
    --cache-dir {CACHE}/perchunk --concurrency {CONCURRENCY}

In [ ]:
!python scripts/evaluate_cv.py --data {DATA} --folds {FOLDS} \
    --scores {OUT}/perchunk/scores.jsonl \
    --score-expr "m3.max_chunk_score" \
    --faith-expr "m3.max_chunk_score" \
    --output {OUT}/perchunk/report.json

In [ ]:
RUNS["Г3 perchunk"] = {"scores": f"{OUT}/perchunk/scores.jsonl",
                       "report": f"{OUT}/perchunk/report.json",
                       "score": lambda sc: sc["m3.max_chunk_score"]}
row = report_row(RUNS["Г3 perchunk"]["report"], "Г3 perchunk")
display(pd.DataFrame([row]))

print(f"\nГ3: recall unreliable {row['recall unrel']} при precision {row['precision unrel']}; "
      f"цель — recall ≥{TARGET_RECALL_UNRELIABLE} при precision ≥{TARGET_PRECISION_UNRELIABLE} — "
      f"{'ЦЕЛЬ ВЗЯТА' if row['recall unrel'] >= TARGET_RECALL_UNRELIABLE and row['precision unrel'] >= TARGET_PRECISION_UNRELIABLE else 'цель не взята'}")

# Целевая фича ветки: расхождение вердиктов между чанками как детектор склейки фактов.
pc = load_scores(RUNS["Г3 perchunk"]["scores"])
evaluated = [s for s in samples if s.id in pc]
for key in ("m3.chunk_disagreement", "m3.n_supporting", "m3.max_chunk_score"):
    point, lo, hi = auc_ci([1 - int(s.reliable) for s in evaluated],
                           [pc[s.id][key] for s in evaluated])
    print(f"AUC({key}) против unreliable: {point:.3f} [{lo:.3f}; {hi:.3f}]")

mixup = [s for s in evaluated if s.marker == "reason_chunk_fact_mixup"]
if mixup:
    rest = [s for s in evaluated if s.marker != "reason_chunk_fact_mixup"]
    print(f"\nchunk_disagreement на {len(mixup)} кейсах с маркером chunk_fact_mixup: "
          f"{np.mean([pc[s.id]['m3.chunk_disagreement'] for s in mixup]):.3f} "
          f"против {np.mean([pc[s.id]['m3.chunk_disagreement'] for s in rest]):.3f} на остальных")

## H5 · GEPA — маркеры как текстовый feedback

**Гипотеза (общепроектная).** При промпт-оптимизации текстовый feedback вида «ответ
содержит факт, которого нет в чанках → `reason_hallucinated_fact`» несёт больше
сигнала, чем бинарное «верно/неверно», и при равном бюджете даёт лучший промпт.

**Статус.** Прошлый прогон остановлен по pre-registered стоп-правилу, но апостериорно
разница оказалась неотличима от нуля: ДИ приращения [−0.037; +0.100]. Это значит, что
гипотеза **осталась непроверенной, а не опровергнутой** — прогон повторяется с
исправленной метрикой (balanced accuracy вместо accuracy: при 72/28 accuracy
вознаграждала промпт, который всегда говорит PASS) и D_pareto без утечки held-out.

Дизайн: 3 сида × 2 варианта (`markers`, `plain`), равный бюджет `--auto medium`,
парный бутстрэп по кейсам. Стоп-правило: H5 отвергается **только** если верхняя
граница 95% ДИ приращения ниже нуля.

**~1.2 ч на прогон, ~7 ч на всё** — только через DataSphere Jobs: VM ноутбука
останавливается при простое.

In [ ]:
# Задания уже описаны в jobs/. Отправлять из терминала DataSphere CLI, не из ячейки:
# задание живёт дольше сессии ноутбука.
for variant in ("markers", "plain"):
    for seed in (0, 1, 2):
        print(f"datasphere project job execute -p <project-id> -c jobs/gepa_{variant}_seed{seed}.yaml")
print("\nКаждое задание поднимает свой vLLM (jobs/_with_vllm.sh) и кладёт эволюционировавший "
      "промпт в artifacts/gepa/<variant>_seed<seed>/m3_gepa_prompt_faithfulness_*.txt")

In [ ]:
# Когда задания досчитали: прогнать судью с каждым эволюционировавшим промптом.
# Промпт подаётся в run_m3.py без преобразований — тем же текстом, каким его сохранил GEPA.
for variant in ("markers", "plain"):
    for seed in (0, 1, 2):
        prompt = f"artifacts/gepa/{variant}_seed{seed}/m3_gepa_prompt_faithfulness_{variant}_seed{seed}.txt"
        if not os.path.isfile(prompt):
            print(f"нет {prompt} — задание ещё не досчитало, пропуск")
            continue
        !python scripts/run_m3.py --data {DATA} \
            --output {OUT}/gepa_{variant}_s{seed}/scores.jsonl \
            --run-meta {OUT}/gepa_{variant}_s{seed}/run.yaml \
            --mode gepa --backend openai_judge --prompt-style axes \
            --prompt-file-faithfulness {prompt} \
            --model {MODEL} --api-base {API_BASE} \
            --cache-dir {CACHE}/gepa_{variant}_s{seed} --concurrency {CONCURRENCY}
        !python scripts/evaluate_cv.py --data {DATA} --folds {FOLDS} \
            --scores {OUT}/gepa_{variant}_s{seed}/scores.jsonl \
            --score-expr "m3.p_faith * m3.p_rel" \
            --faith-expr "m3.p_faith" --rel-expr "m3.p_rel" \
            --compare {OUT}/axes/scores.jsonl \
            --output {OUT}/gepa_{variant}_s{seed}/report.json

In [ ]:
# Вердикт по H5 считает сам CLI: парный бутстрэп по кейсам, оба плеча по трём сидам.
!python scripts/run_gepa.py --mode h5 --data {DATA} --folds {FOLDS} \
    --h5-markers {OUT}/gepa_markers_s0/scores.jsonl \
    --h5-markers {OUT}/gepa_markers_s1/scores.jsonl \
    --h5-markers {OUT}/gepa_markers_s2/scores.jsonl \
    --h5-plain {OUT}/gepa_plain_s0/scores.jsonl \
    --h5-plain {OUT}/gepa_plain_s1/scores.jsonl \
    --h5-plain {OUT}/gepa_plain_s2/scores.jsonl \
    --output-dir results/gepa

In [ ]:
h5_path = "results/gepa/h5_markers_vs_plain.json"
if os.path.isfile(h5_path):
    h5 = json.load(open(h5_path, encoding="utf-8"))
    print(json.dumps(h5, ensure_ascii=False, indent=2)[:2000])
    print("\nСтоп-правило: H5 отвергается только если верхняя граница ДИ приращения < 0. "
          "ДИ, накрывающий ноль, означает «не проверено», а не «опровергнуто».")
else:
    print(f"нет {h5_path} — плечи ещё не досчитаны")

## H1 · Fine-tuned судья против prompt-optimized

**Гипотеза (совместная с веткой метода 1).** Fine-tuned судья превосходит
prompt-optimized по macro-F1 на `reliable`. Наш вклад — правая часть сравнения; чтобы
сравнение было честным, backbone тот же (`Qwen2.5-7B-Instruct`) и схема извлечения
вероятностей та же (logprobs `PASS`/`FAIL`).

Конфигурация: LoRA all-linear, r=256, α=512, lr 2e-4, 3 эпохи, `max_length` 2048.
Взвешенный лосс (`--pos-weight-mode balanced`) и oversampling негативов — обязательны:
прошлый прогон на 1.5B схлопнулся в константный вердикт (1,1) из-за дисбаланса 72/28.
`--save-strategy epoch`: при `no` обрыв стоит всего многочасового прогона.

Обучение идёт **по фолдам** — предсказания собираются out-of-fold, иначе судья
отчитывается на данных, которые видел. **~1.5 ч на фолд, ~7.5 ч на пять** — Jobs.

In [ ]:
for fold in range(5):
    print(f"datasphere project job execute -p <project-id> -c jobs/ft_judge_fold{fold}.yaml")
print("\nКаждое задание кладёт scores.jsonl фолда в predictions/alfa/ft_judge/direct_fold<k>/ "
      "и диагностику схлопывания в ft_diagnostics.json")

In [ ]:
# Контроль схлопывания — до метрик: константный вердикт даёт правдоподобный F1 и
# бессмысленную модель.
for fold in range(5):
    path = f"predictions/alfa/ft_judge/direct_fold{fold}/ft_diagnostics.json"
    if os.path.isfile(path):
        diagnostics = json.load(open(path, encoding="utf-8"))
        print(f"fold {fold}: {json.dumps(diagnostics, ensure_ascii=False)[:400]}")
    else:
        print(f"fold {fold}: нет {path} — задание ещё не досчитало")

In [ ]:
# Пять фолдов склеиваются в один OOF-артефакт: каждый кейс предсказан моделью,
# которая его не видела.
parts = [f"predictions/alfa/ft_judge/direct_fold{fold}/scores.jsonl" for fold in range(5)]
if all(os.path.isfile(part) for part in parts):
    os.makedirs("predictions/alfa/ft_judge/oof", exist_ok=True)
    seen = set()
    with open("predictions/alfa/ft_judge/oof/scores.jsonl", "w", encoding="utf-8") as out:
        for part in parts:
            for line in open(part, encoding="utf-8"):
                if not line.strip():
                    continue
                row_id = json.loads(line)["id"]
                assert row_id not in seen, (
                    f"кейс {row_id} предсказан дважды: фолды пересекаются, OOF-склейка неверна"
                )
                seen.add(row_id)
                out.write(line)
    print(f"OOF собран: {len(seen)} кейсов")

    !python scripts/evaluate_cv.py --data {DATA} --folds {FOLDS} \
        --scores predictions/alfa/ft_judge/oof/scores.jsonl \
        --score-expr "m3.p_faith * m3.p_rel" \
        --faith-expr "m3.p_faith" --rel-expr "m3.p_rel" \
        --compare {OUT}/axes_sc8/scores.jsonl \
        --output predictions/alfa/ft_judge/oof/report.json
else:
    print("не все фолды досчитаны — склейка пропущена")

## Итоговые метрики

Одна таблица на все прогоны. `выше шума` — сравнение с 500 нулевыми прогонами
(перестановка меток): значение ниже 95-го перцентиля шума означает, что метрика
неотличима от случайной, каким бы правдоподобным ни выглядело само число.

Сравнивать точечные оценки без интервалов бессмысленно: на 2233 кейсах ширина ДИ у
macro-F1 составляет около ±0.06, и два «разных» результата в этих пределах — один и
тот же результат.

In [ ]:
if os.path.isfile("predictions/alfa/ft_judge/oof/report.json"):
    RUNS["H1 FT-судья (OOF)"] = {"scores": "predictions/alfa/ft_judge/oof/scores.jsonl",
                                 "report": "predictions/alfa/ft_judge/oof/report.json",
                                 "score": JOINT_SCORE}
for variant in ("markers", "plain"):
    path = f"{OUT}/gepa_{variant}_s0/report.json"
    if os.path.isfile(path):
        RUNS[f"H5 GEPA {variant} s0"] = {"scores": f"{OUT}/gepa_{variant}_s0/scores.jsonl",
                                         "report": path, "score": JOINT_SCORE}

summary = pd.DataFrame([report_row(run["report"], label)
                        for label, run in RUNS.items() if os.path.isfile(run["report"])])
summary = summary.sort_values("macro-F1", ascending=False).reset_index(drop=True)
display(summary)

In [ ]:
diagnostics = pd.DataFrame([judge_diagnostics(run["scores"], label)
                            for label, run in RUNS.items() if os.path.isfile(run["scores"])])
display(diagnostics)

In [ ]:
# Опорные числа для сравнения: чем этот контур мерялся раньше и что даёт surface-бейзлайн.
# Когорта у опорных чисел разная, и без неё они несопоставимы: стэк считался на
# пересечении покрытий источников (331 кейс), остальное — на полных 1480 из фолдов.
REFERENCE = [
    ("константа «всегда reliable»", 1480, "0.4203 — потолок бессмысленной модели"),
    ("surface, полная когорта",     1480, "0.5350 [0.5102; 0.5615]"),
    ("surface, когорта стэка",       331, "0.5791 [0.5222; 0.6348]"),
    ("surface + m3.p_faith, стэк",   331, "0.6543 [0.5956; 0.7097], Δ +0.0752 [+0.0158; +0.1344], p=0.013"),
]
for name, cohort, value in REFERENCE:
    print(f"{name:30} n={cohort:5}   {value}")
print("\nСтроки с разным n сравнивать между собой нельзя — только внутри своей когорты.")

In [ ]:
# Вердикты по гипотезам — по числам, посчитанным выше, без пересчёта.
best = summary.iloc[0]
print(f"Лучший вариант: {best['вариант']}  macro-F1 {best['macro-F1']} {best['95% ДИ']}, "
      f"выше шума: {best['выше шума']}\n")

for _, row in summary.iterrows():
    base = summary[summary["вариант"] == "Г0 holistic"]
    delta = row["macro-F1"] - float(base["macro-F1"].iloc[0]) if len(base) else float("nan")
    print(f"{row['вариант']:22} macro-F1 {row['macro-F1']:.4f} {row['95% ДИ']}  "
          f"Δ к базе {delta:+.4f}  recall unrel {row['recall unrel']}")
print("\nΔ без парного ДИ — не результат: ДИ приращения печатает evaluate_cv по --compare, "
      "смотри поле comparisons в report.json соответствующего прогона.")

In [ ]:
# Финальный срез в стиле baseline организаторов: как лучший вариант распределяет ошибки
# по классам при пороге, подобранном протоколом.
rep = json.load(open(RUNS[best["вариант"]]["report"], encoding="utf-8"))
threshold = rep["diagnostics"]["operating_threshold"]
rows = load_scores(RUNS[best["вариант"]]["scores"])
# Та же когорта, что и в report.json: кейсы, попавшие в folds.assignment. 753 id
# гигантской группы из CV исключены, и считать по ним срез — значит отчитываться
# на кейсах, которых нет в метрике выше.
evaluated = [s for s in samples if s.id in rows and s.id in folds["assignment"]]
assert len(evaluated) == best["n"], (
    f"когорта среза {len(evaluated)} не совпала с n_evaluated {best['n']} из отчёта"
)

score_of = RUNS[best["вариант"]]["score"]   # то же выражение, что ушло в --score-expr
y_true = [int(s.reliable) for s in evaluated]
y_pred = [int(score_of(rows[s.id]) >= threshold) for s in evaluated]

print(f"порог {threshold:.3f} (подобран внутри train-части фолда, не на этих кейсах)\n")
print(classification_report(y_true, y_pred, target_names=["unreliable", "reliable"], digits=4))

In [ ]:
# Коммит артефактов: прогон не должен зависеть от того, доживёт ли сессия.
!git add predictions/alfa/m3_judge predictions/alfa/ft_judge results/gepa 2>/dev/null; \
 git commit -m "results(m3): прогоны судьи по гипотезам Г0-Г3, H5, H1" && git log --oneline -1